# Genre2Vec — Data to Embeddings
Reads `songs.csv` and `tags.csv`, trains skip-gram, saves `genre_embeddings.json`.

In [ ]:
import numpy as np
import os
import json
from collections import defaultdict
import torch
import torch.nn as nn

## Config

In [ ]:
DATA_DIR = './data/'
CSV_DIR = DATA_DIR + './csv/'
SONGS_PATH = CSV_DIR + "songs.csv"
TAGS_PATH = CSV_DIR + "tags.csv"
OUTPUT_PATH = DATA_DIR + "/embeddings/genre_embeddings.json"
MODELS_DIR = "./models/"

MIN_FREQ = 3
WINDOW = 4
EMBED_DIM = 64
NEG_SAMPLES = 200
LR = 0.025
BATCH_SIZE = 1024
TAG_POP_CAP = 5

BATCH_SIZE = 4096
EPOCHS = 200
PATIENCE   = 5
MIN_DELTA  = 0.001

In [ ]:
paths = [SONGS_PATH, TAGS_PATH]
dirs = [DATA_DIR, CSV_DIR]
for d in dirs:
    if not os.path.exists(d):
        os.makedirs(d)

for p in paths:
    assert os.path.exists(p), f"Missing file: {p}"

os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

## 1. Load & Clean Data

In [ ]:
def read_csv(path):
    with open(path) as f:
        lines = f.read().strip().splitlines()
    headers = [h.strip('"') for h in lines[0].split(',')]
    rows = []
    for line in lines[1:]:
        parts = [p.strip('"') for p in line.split(';')]
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))
    return rows

docs = defaultdict(set)
for row in read_csv(SONGS_PATH):
    docs[row["spotify_id"]].add(row["genre_name"])

for row in read_csv(TAGS_PATH):
    weight = min(int(row["popularity"]), TAG_POP_CAP)
    for _ in range(weight):
        docs[row["song_spotify_id"]].add(row["tag"])

documents = [list(tokens) for tokens in docs.values() if len(tokens) >= 2]
print(f"{len(documents):,} songs | {sum(len(d) for d in documents):,} total tokens")

## 2. Vocabulary

In [ ]:
freq = defaultdict(int)
for doc in documents:
    for t in doc: freq[t] += 1

idx2token = [t for t, c in sorted(freq.items()) if c >= MIN_FREQ]
token2idx = {t: i for i, t in enumerate(idx2token)}
V = len(idx2token)
print(f"{V} tokens in vocab")

## 3. Skip-gram Pairs

In [ ]:
pairs = []
for doc in [[token2idx[t] for t in d if t in token2idx] for d in documents]:
    if len(doc) < 2: continue
    for i, c in enumerate(doc):
        for j in range(max(0, i - WINDOW), min(len(doc), i + WINDOW + 1)):
            if j != i: pairs.append((c, doc[j]))

pairs = np.array(pairs, dtype=np.int32)
print(f"{len(pairs):,} training pairs")

## 4. Train Genre2Vec

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

class Genre2Vec(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.W_in  = nn.Embedding(vocab_size, embed_dim)
        self.W_out = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.W_in.weight,  -0.1, 0.1)
        nn.init.constant_(self.W_out.weight, 0)

    def forward(self, center, context, negatives):
        vc = self.W_in(center)
        vo = self.W_out(context)
        vn = self.W_out(negatives)
        pos_loss = torch.log(torch.sigmoid((vc * vo).sum(1))).mean()
        neg_loss = torch.log(torch.sigmoid(-(vc.unsqueeze(1) * vn).sum(2))).mean()
        return -(pos_loss + neg_loss)

model = Genre2Vec(V, EMBED_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
pairs_tensor = torch.tensor(pairs, dtype=torch.long)

best_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    idx = torch.randperm(len(pairs_tensor))
    pairs_shuffled = pairs_tensor[idx]
    total_loss = 0.0

    for s in range(0, len(pairs_shuffled), BATCH_SIZE):
        batch = pairs_shuffled[s:s+BATCH_SIZE].to(device)
        center    = batch[:, 0]
        context   = batch[:, 1]
        negatives = torch.randint(0, V, (len(batch), NEG_SAMPLES), device=device)

        loss = model(center, context, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / (len(pairs_shuffled) // BATCH_SIZE)
    print(f"Epoch {epoch+1}/{EPOCHS} — loss={avg_loss:.4f}")

    if avg_loss < best_loss - MIN_DELTA:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODELS_DIR + "embedding_model.pt")
    else:
        patience_counter += 1
        print(f"  no improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(torch.load( MODELS_DIR + "embedding_model.pt"))
Win = model.W_in.weight.detach().cpu().numpy()
print(f"Done. Best loss: {best_loss:.4f}")

## 5. Save Embeddings as JSON

In [ ]:
norms = np.linalg.norm(Win, axis=1, keepdims=True) + 1e-9
Win_norm = Win / norms

embeddings = {token: Win_norm[i].tolist() for i, token in enumerate(idx2token)}

with open(OUTPUT_PATH, "w") as f:
    json.dump(embeddings, f)

print(f"Saved {len(embeddings)} embeddings to {OUTPUT_PATH}")